# VoiceForge

Inspect prepared speech manifests, speaker balance, and the local training path for the SpeechT5 voice-cloning service.

In [ ]:
from pathlib import Path
import json
import pandas as pd

service_dir = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
data_dir = service_dir / 'data' / 'voiceforge'
processed_dir = data_dir / 'processed'
processed_dir

In [ ]:
summary_path = processed_dir / 'summary.json'
summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
summary

In [ ]:
train_path = processed_dir / 'train_manifest.jsonl'
eval_path = processed_dir / 'eval_manifest.jsonl'

def load_jsonl(path: Path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

train_rows = load_jsonl(train_path)
eval_rows = load_jsonl(eval_path)
len(train_rows), len(eval_rows)

In [ ]:
train_df = pd.DataFrame(train_rows)
eval_df = pd.DataFrame(eval_rows)
train_df.head()

In [ ]:
if not train_df.empty:
    display(train_df.groupby(['source', 'speaker_id']).size().reset_index(name='clips').sort_values('clips', ascending=False).head(20))
else:
    print('Run scripts/download_data.py and scripts/prepare_dataset.py first.')

## Useful commands

```bash
cd /Users/toxa/git/playground/src/voiceforge
python scripts/download_data.py
python scripts/prepare_dataset.py --max-per-speaker 200
python scripts/train_model.py --device mps --epochs 1 --max-train-samples 64 --max-eval-samples 16
python main.py
```

## Latest local run

Inspect the most recent fine-tuning artifact and generated preview clips from `models/speecht5-finetuned`.


In [ ]:
from IPython.display import Audio, display

model_dir = service_dir / "models" / "speecht5-finetuned"
artifact_path = model_dir / "artifact.json"
preview_manifest_path = model_dir / "previews" / "preview_manifest.json"

artifact = json.loads(artifact_path.read_text()) if artifact_path.exists() else {}
preview_manifest = json.loads(preview_manifest_path.read_text()) if preview_manifest_path.exists() else []
artifact


In [ ]:
preview_df = pd.DataFrame(preview_manifest)
preview_df


In [ ]:
for row in preview_manifest:
    generated_path = Path(row["generated_audio"])
    print(f"{row["speaker_id"]} | {row["source"]} | {generated_path.name}")
    if generated_path.exists():
        display(Audio(filename=str(generated_path)))
    else:
        print("Missing generated preview:", generated_path)
